# CNN-v2 Final Benchmark

Runs the exact 10-cycle CNN reference plus two regularized CNN-v2 candidates on the same promoter `train`, `eval`/validation, and `test` CSV splits. If CNN-v2 does not improve validation MCC and held-out test MCC/AUPRC, keep the reference CNN and move to DNABERT2.


## 1. Set Up SeqTrainer In Colab

Clone the branch, install SeqTrainer, and verify imports.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-model-baselines-clean"
REPO_DIR = Path("/content/SeqTrainer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}; updating {BRANCH}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("Python import path includes:", SRC_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[torch]"], check=True)

import seqtrainer
print("SeqTrainer import OK:", seqtrainer.__file__)

## 2. Prepare Dataset Files

Use Drive CSVs when available; otherwise extract the bundled repo ZIP into `data/promoter_classification/`.

In [ ]:
from pathlib import Path
import shutil
import zipfile

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except ValueError as exc:
        print(f"Google Drive mount failed: {exc}")
        print("Continuing without Drive. The notebook will use local CSVs or the bundled repo ZIP if available.")
except ModuleNotFoundError:
    print("google.colab is not available. Assuming dataset files are already local or in the repo zip.")

LOCAL_DATA_DIR = Path("data/promoter_classification")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

split_file_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

expected_relative_dir = Path("AIxBio") / "Promoter Classification" / "Data"
drive_roots = []
if DRIVE_MOUNTED:
    drive_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive/Shareddrives"),
    ]


def has_all_split_files(directory: Path) -> bool:
    return all((directory / file_name).exists() for file_name in split_file_names.values())


def find_drive_data_dir() -> Path | None:
    candidates = [root / expected_relative_dir for root in drive_roots]
    print("Checking expected Drive paths:")
    for candidate in candidates:
        print(f"  - {candidate}")
        if has_all_split_files(candidate):
            return candidate

    search_roots = []
    for root in drive_roots:
        aixbio = root / "AIxBio"
        if aixbio.exists():
            search_roots.append(aixbio)
    for root in search_roots:
        print(f"Searching for train CSV under: {root}")
        for train_path in root.rglob(split_file_names["train"]):
            candidate = train_path.parent
            if has_all_split_files(candidate):
                return candidate
    return None


def copy_from_drive(data_dir: Path) -> bool:
    for split, file_name in split_file_names.items():
        source = data_dir / file_name
        target = LOCAL_DATA_DIR / file_name
        shutil.copy2(source, target)
        print(f"Copied {split}: {target}")
    return True


def extract_from_repo_zip() -> bool:
    zip_path = Path("data/data_DNABERT/promoter_classification_DNABERT.zip")
    if not zip_path.exists():
        return False

    print(f"Drive CSVs were not found locally. Extracting from repo zip: {zip_path}")
    with zipfile.ZipFile(zip_path) as zf:
        members = set(zf.namelist())
        for split, file_name in split_file_names.items():
            if file_name not in members:
                raise FileNotFoundError(f"{file_name} is missing from {zip_path}")
            target = LOCAL_DATA_DIR / file_name
            with zf.open(file_name) as source, target.open("wb") as dest:
                shutil.copyfileobj(source, dest)
            print(f"Extracted {split}: {target}")
    return True


drive_data_dir = find_drive_data_dir()
if drive_data_dir is not None:
    print(f"Using Drive data directory: {drive_data_dir}")
    copy_from_drive(drive_data_dir)
elif all((LOCAL_DATA_DIR / file_name).exists() for file_name in split_file_names.values()):
    print(f"Using existing local CSV files in: {LOCAL_DATA_DIR}")
elif extract_from_repo_zip():
    print(f"Using CSV files extracted into: {LOCAL_DATA_DIR}")
else:
    raise FileNotFoundError(
        "Could not find the promoter CSV files in Drive or in the repo zip. "
        "Expected Drive folder: /content/drive/MyDrive/AIxBio/Promoter Classification/Data"
    )

## 3. Verify Data And Benchmark Contract

Confirm source, split files, label balance, and sequence lengths before training.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/SeqTrainer")
SRC_DIR = REPO_DIR / "src"
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
from seqtrainer.benchmarks import load_benchmark_config

CONFIG_PATH = Path("config-examples/benchmarks/cnn.toml")
config = load_benchmark_config(CONFIG_PATH)

print("Dataset:", config.dataset.name)
print("Source accession:", config.dataset.source_accession)
print("Split strategy:", config.split.strategy)
print("Primary metric:", config.evaluation.primary_metric)

IMBALANCE_RATIO_THRESHOLD = 1.5
split_balance_rows = []
TRAIN_SPLIT_IMBALANCE_RATIO = None

for split, csv_path in config.dataset.split_files.items():
    frame = pd.read_csv(csv_path)
    label_counts = frame[config.dataset.label_field].value_counts().sort_index().to_dict()
    sequence_lengths = frame[config.dataset.sequence_field].astype(str).str.len()
    majority_count = max(label_counts.values())
    minority_count = min(label_counts.values())
    imbalance_ratio = majority_count / max(minority_count, 1)
    balance_note = "imbalanced" if imbalance_ratio >= IMBALANCE_RATIO_THRESHOLD else "balanced"
    split_balance_rows.append(
        {
            "split": split,
            "rows": len(frame),
            "label_counts": label_counts,
            "imbalance_ratio": imbalance_ratio,
            "balance_note": balance_note,
        }
    )
    if split == "train":
        TRAIN_SPLIT_IMBALANCE_RATIO = imbalance_ratio
    print(
        f"{split}: rows={len(frame)} labels={label_counts} "
        f"imbalance_ratio={imbalance_ratio:.3f} balance={balance_note} "
        f"length_min={sequence_lengths.min()} length_max={sequence_lengths.max()} "
        f"length_mean={sequence_lengths.mean():.1f}"
    )

USE_CLASS_WEIGHTING = TRAIN_SPLIT_IMBALANCE_RATIO >= IMBALANCE_RATIO_THRESHOLD
print("Use class weighting for CNN-v2 experiments:", USE_CLASS_WEIGHTING)
display(pd.DataFrame(split_balance_rows))

## 4. Define CNN Improvement Experiments

Run only the reference model and two final CNN-v2 candidates. Class weighting activates only if the training split is meaningfully imbalanced.


In [ ]:
import torch
from seqtrainer.torch.cnn_baseline import CnnCsvSplitConfig, run_cnn_csv_splits

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEQUENCE_LENGTH = config.preprocessing.sequence_length or 300
BASE_OUTPUT_DIR = Path("outputs/cnn_v2_final_benchmark")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Sequence length:", SEQUENCE_LENGTH)

EXPERIMENTS = [
    {
        "name": "tiny_10_cycles_reference",
        "model_variant": "tiny",
        "cycles": 10,
        "batch_size": 16,
        "learning_rate": 1e-3,
        "weight_decay": 0.0,
        "optimizer_name": "adam",
        "scheduler_name": "none",
        "select_best_by_mcc": False,
        "early_stopping_patience": None,
        "dropout": 0.25,
        "class_weighting": False,
    },
    {
        "name": "cnn_v2_regularized_50_cycles",
        "model_variant": "enhanced",
        "cycles": 50,
        "batch_size": 32,
        "learning_rate": 3e-4,
        "weight_decay": 2e-4,
        "optimizer_name": "adamw",
        "scheduler_name": "one_cycle",
        "select_best_by_mcc": True,
        "early_stopping_patience": 10,
        "dropout": 0.30,
        "class_weighting": USE_CLASS_WEIGHTING,
    },
    {
        "name": "cnn_v2_regularized_100_cycles",
        "model_variant": "enhanced",
        "cycles": 100,
        "batch_size": 32,
        "learning_rate": 3e-4,
        "weight_decay": 2e-4,
        "optimizer_name": "adamw",
        "scheduler_name": "one_cycle",
        "select_best_by_mcc": True,
        "early_stopping_patience": 12,
        "dropout": 0.30,
        "class_weighting": USE_CLASS_WEIGHTING,
    },
]

# For a very fast smoke test, set this to 1. Use None for the final CNN-v2 run.
MAX_EXPERIMENTS = None
experiments_to_run = EXPERIMENTS if MAX_EXPERIMENTS is None else EXPERIMENTS[:MAX_EXPERIMENTS]

pd.DataFrame(experiments_to_run)

## 5. Run The Final CNN-v2 Candidates

Each run writes metrics, history, manifest, and predictions under `outputs/cnn_v2_final_benchmark/<experiment_name>/`.

In [ ]:
all_metrics = []
all_histories = []

for exp in experiments_to_run:
    print()
    print("=== Running", exp["name"], "===")
    output_dir = BASE_OUTPUT_DIR / exp["name"]

    run_config = CnnCsvSplitConfig(
        train_csv=config.dataset.split_files["train"],
        validation_csv=config.dataset.split_files["validation"],
        test_csv=config.dataset.split_files["test"],
        output_dir=output_dir,
        dataset_name=config.dataset.name,
        source_accession=config.dataset.source_accession,
        source_url=config.dataset.source_url,
        sequence_field=config.dataset.sequence_field,
        label_field=config.dataset.label_field,
        sequence_length=SEQUENCE_LENGTH,
        seed=config.experiment.seed,
        batch_size=exp["batch_size"],
        cycles=exp["cycles"],
        learning_rate=exp["learning_rate"],
        weight_decay=exp["weight_decay"],
        optimizer_name=exp["optimizer_name"],
        scheduler_name=exp["scheduler_name"],
        select_best_by_mcc=exp["select_best_by_mcc"],
        early_stopping_patience=exp["early_stopping_patience"],
        model_variant=exp["model_variant"],
        dropout=exp["dropout"],
        class_weighting=exp["class_weighting"],
        device=DEVICE,
    )

    result = run_cnn_csv_splits(run_config)
    metrics_df = pd.read_csv(Path(result.output_dir) / "metrics.csv")
    history_df = pd.read_csv(Path(result.output_dir) / "history.csv")
    metrics_df.insert(0, "experiment", exp["name"])
    history_df.insert(0, "experiment", exp["name"])
    all_metrics.append(metrics_df)
    all_histories.append(history_df)

summary_df = pd.concat(all_metrics, ignore_index=True)
history_df = pd.concat(all_histories, ignore_index=True)
summary_path = BASE_OUTPUT_DIR / "summary_metrics.csv"
history_path = BASE_OUTPUT_DIR / "summary_history.csv"
summary_df.to_csv(summary_path, index=False)
history_df.to_csv(history_path, index=False)
print("Wrote:", summary_path)
print("Wrote:", history_path)

## 6. Compare Full Metrics

Choose candidates by validation MCC. Use held-out test MCC and AUPRC only after the candidate is selected.

In [ ]:
metric_cols = [
    "experiment",
    "split",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "mcc",
    "sensitivity",
    "specificity",
    "tn",
    "fp",
    "fn",
    "tp",
    "auroc",
    "auprc",
    "loss",
]
existing_cols = [col for col in metric_cols if col in summary_df.columns]

validation_rank = summary_df[summary_df["split"] == "validation"].sort_values(
    ["mcc", "auprc", "balanced_accuracy"], ascending=False
)
test_rank = summary_df[summary_df["split"] == "test"].sort_values(
    ["mcc", "auprc", "balanced_accuracy"], ascending=False
)

print("Validation ranking: choose model here")
display(validation_rank[existing_cols])

print("Test ranking: report after validation choice")
display(test_rank[existing_cols])

## 7. Write The Issue 3 CNN Decision Record

Write a small JSON decision record for whether CNN-v2 should replace the reference CNN.

In [ ]:
import json

REFERENCE_EXPERIMENT = "tiny_10_cycles_reference"
MIN_MEANINGFUL_VALIDATION_MCC_GAIN = 0.02

validation_metrics = summary_df[summary_df["split"] == "validation"].set_index("experiment")
test_metrics = summary_df[summary_df["split"] == "test"].set_index("experiment")
selected_experiment = str(validation_rank.iloc[0]["experiment"])

reference_validation_mcc = float(validation_metrics.loc[REFERENCE_EXPERIMENT, "mcc"])
selected_validation_mcc = float(validation_metrics.loc[selected_experiment, "mcc"])
validation_mcc_delta = selected_validation_mcc - reference_validation_mcc

reference_test_mcc = float(test_metrics.loc[REFERENCE_EXPERIMENT, "mcc"])
selected_test_mcc = float(test_metrics.loc[selected_experiment, "mcc"])
test_mcc_delta = selected_test_mcc - reference_test_mcc

reference_test_auprc = float(test_metrics.loc[REFERENCE_EXPERIMENT, "auprc"])
selected_test_auprc = float(test_metrics.loc[selected_experiment, "auprc"])
test_auprc_delta = selected_test_auprc - reference_test_auprc

use_improved_cnn = (
    selected_experiment != REFERENCE_EXPERIMENT
    and validation_mcc_delta >= MIN_MEANINGFUL_VALIDATION_MCC_GAIN
    and test_mcc_delta > 0
    and test_auprc_delta >= 0
)

decision = {
    "reference_experiment": REFERENCE_EXPERIMENT,
    "selected_by_validation_mcc": selected_experiment,
    "minimum_meaningful_validation_mcc_gain": MIN_MEANINGFUL_VALIDATION_MCC_GAIN,
    "validation_mcc_delta_vs_reference": validation_mcc_delta,
    "test_mcc_delta_vs_reference": test_mcc_delta,
    "test_auprc_delta_vs_reference": test_auprc_delta,
    "recommendation": "use_improved_cnn_candidate" if use_improved_cnn else "keep_reference_cnn_and_move_to_dnabert2",
}

decision_path = BASE_OUTPUT_DIR / "issue3_cnn_decision.json"
decision_path.write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps(decision, indent=2))
print("Wrote:", decision_path)

## 8. Inspect Training Curves

Inspect train/validation loss curves for overfitting.

In [ ]:
display(history_df.groupby("experiment").tail(5))

try:
    import matplotlib.pyplot as plt

    for exp_name, group in history_df.groupby("experiment"):
        plt.figure(figsize=(7, 4))
        plt.plot(group["cycle"], group["train_loss"], label="train_loss")
        plt.plot(group["cycle"], group["validation_loss"], label="validation_loss")
        plt.title(exp_name)
        plt.xlabel("cycle")
        plt.ylabel("loss")
        plt.legend()
        plt.show()
except Exception as exc:
    print("Plotting skipped:", exc)

## 9. Optional: Save CNN-v2 Outputs Back To Drive

Enable this only when Google Drive is mounted.

In [ ]:
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Outputs/cnn_v2_final_benchmark")

if SAVE_TO_DRIVE and DRIVE_MOUNTED:
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for source in BASE_OUTPUT_DIR.rglob("*"):
        if source.is_file():
            target = DRIVE_OUTPUT_ROOT / source.relative_to(BASE_OUTPUT_DIR)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    print("Saved outputs to:", DRIVE_OUTPUT_ROOT)
elif SAVE_TO_DRIVE:
    print("Drive is not mounted; outputs remain local at:", BASE_OUTPUT_DIR)